In [7]:
# ── Cell 1: Imports & Utility Functions ─────────────────────────────────────
# Run once per session.

import os
import tempfile
from pathlib import Path

import numpy as np

import pybullet as p
import pybullet_data

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


def resolve_package_urdf(src_urdf: str, desc_root: str) -> str:
    """Replace package://aloha_new_description/... with absolute mesh root path."""
    txt = Path(src_urdf).read_text(encoding='utf-8')
    txt = txt.replace(
        'package://aloha_new_description/',
        str(Path(desc_root).as_posix().rstrip('/')) + '/'
    )
    fd, tmp = tempfile.mkstemp(prefix='aloha_resolved_', suffix='.urdf')
    os.close(fd)
    Path(tmp).write_text(txt, encoding='utf-8')
    return tmp


def load_actions_npy(path: str, expected_dim: int = 14) -> np.ndarray:
    """Load action trajectory from .npy file and return float32 [T, D]."""
    if not str(path).endswith('.npy'):
        raise ValueError(f'{path}: only .npy is supported')
    arr = np.load(path)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f'{path}: expected [T, D], got {arr.shape}')
    if arr.shape[1] != expected_dim:
        raise ValueError(f'{path}: expected D={expected_dim}, got D={arr.shape[1]}')
    return arr.astype(np.float32)


def pad_actions_edge(x: np.ndarray, T: int) -> np.ndarray:
    if len(x) == T:
        return x
    if len(x) == 0:
        raise ValueError('Empty trajectory is not supported')
    if len(x) > T:
        return x[:T]
    pad_n = T - len(x)
    return np.concatenate([x, np.repeat(x[-1:], pad_n, axis=0)], axis=0)


def arm_joint_names(prefix: str):
    return [f'{prefix}_joint{i}' for i in range(1, 9)]


def name_to_joint_index(robot_id: int):
    out = {}
    for j in range(p.getNumJoints(robot_id)):
        info = p.getJointInfo(robot_id, j)
        out[info[1].decode('utf-8')] = j
    return out


def map_arm_6p1_to_8_ignore_gripper(a7: np.ndarray) -> np.ndarray:
    """
    Ignore gripper visualization completely.
    URDF 8 joints: joint1..6=q1..q6, joint7=0, joint8=0
    """
    return np.array([a7[0], a7[1], a7[2], a7[3], a7[4], a7[5], 0.0, 0.0], dtype=np.float32)


def compute_frame_points(robot_id, name2idx, left_6p1, right_6p1,
                         left_joint_names, right_joint_names):
    left_q8 = map_arm_6p1_to_8_ignore_gripper(left_6p1)
    right_q8 = map_arm_6p1_to_8_ignore_gripper(right_6p1)

    chain_names = left_joint_names + right_joint_names
    chain_vals = np.concatenate([left_q8, right_q8], axis=0)

    for jn, q in zip(chain_names, chain_vals):
        p.resetJointState(robot_id, name2idx[jn], float(q))

    pts = []
    parent = []
    for i, jn in enumerate(chain_names):
        jid = name2idx[jn]
        ls = p.getLinkState(robot_id, jid, computeForwardKinematics=True)
        pts.append(np.array(ls[0], dtype=np.float32))
        parent.append(i - 1 if (i % 8) != 0 else -1)

    return np.stack(pts, axis=0), np.array(parent, dtype=np.int32)


def skeleton_lines(pts: np.ndarray, parent: np.ndarray):
    x, y, z = [], [], []
    for i, pidx in enumerate(parent):
        if pidx < 0:
            continue
        x += [pts[pidx, 0], pts[i, 0], None]
        y += [pts[pidx, 1], pts[i, 1], None]
        z += [pts[pidx, 2], pts[i, 2], None]
    return x, y, z



In [ ]:
# ── Cell 2: Configuration ───────────────────────────────────────────────────
# Edit paths/settings here, then run Cell 3.

# --- URDF ---
URDF_PATH = "/liujinxin/code/lhc/wy/wms/lingbot-va/assets/mobile_aloha_sim/aloha_new_description/urdf/aloha_new.urdf"
ALOHA_DESC_ROOT = "/liujinxin/code/lhc/wy/wms/lingbot-va/assets/mobile_aloha_sim/aloha_new_description" # contains meshes/

ACTION_PREFIX = "/liujinxin/code/lhc/wy/wms/lingbot-va/checkpoints/rc_aloha_pencil_case/bs16lr7e-6_resume8000/12k_local_inf/sample_002_idx563"

# --- Trajectories (.npy only) ---
GT_ACTIONS_NPY = ACTION_PREFIX + "gt_actions_denorm.npy"     # shape [T,14]
PRED_ACTIONS_NPY = ACTION_PREFIX + "pred_actions_denorm.npy"

# --- Which two arms to visualize ---
LEFT_ARM_PREFIX = 'fl'
RIGHT_ARM_PREFIX = 'fr'

# --- Display / placement ---
BASE_POS_GT = [0.0, 0.0, 0.0]
BASE_POS_PRED = [0.0, 0.0, 0.0]
PRED_OFFSET_Y_M = 0.0  # set 0 to align at base
BASE_EULER = [0.0, 0.0, 0.0]

# playback speed for Play widget (ms per frame)
PLAY_INTERVAL_MS = 50

LEFT_ARM_JOINT_NAMES = arm_joint_names(LEFT_ARM_PREFIX)
RIGHT_ARM_JOINT_NAMES = arm_joint_names(RIGHT_ARM_PREFIX)
RESOLVED_URDF_PATH = resolve_package_urdf(URDF_PATH, ALOHA_DESC_ROOT)



In [17]:
# ── Cell 3: Initialize & Visualize GT vs Pred ──────────────────────────────
# Re-run this cell whenever you change Cell 2.

act_gt = load_actions_npy(GT_ACTIONS_NPY, expected_dim=14)
act_pd = load_actions_npy(PRED_ACTIONS_NPY, expected_dim=14)

left_gt  = act_gt[:, 0:7]
right_gt = act_gt[:, 7:14]
left_pd  = act_pd[:, 0:7]
right_pd = act_pd[:, 7:14]

T = max(len(act_gt), len(act_pd))
left_gt  = pad_actions_edge(left_gt, T)
right_gt = pad_actions_edge(right_gt, T)
left_pd  = pad_actions_edge(left_pd, T)
right_pd = pad_actions_edge(right_pd, T)

print(f'GT steps={len(act_gt)}  Pred steps={len(act_pd)}  display T={T}')

try:
    p.disconnect()
except Exception:
    pass
p.connect(p.DIRECT)
p.setAdditionalSearchPath(pybullet_data.getDataPath())

base_pred = [BASE_POS_PRED[0], BASE_POS_PRED[1] + PRED_OFFSET_Y_M, BASE_POS_PRED[2]]
robot_gt = p.loadURDF(
    RESOLVED_URDF_PATH,
    basePosition=BASE_POS_GT,
    baseOrientation=p.getQuaternionFromEuler(BASE_EULER),
    useFixedBase=True,
)
robot_pd = p.loadURDF(
    RESOLVED_URDF_PATH,
    basePosition=base_pred,
    baseOrientation=p.getQuaternionFromEuler(BASE_EULER),
    useFixedBase=True,
)

name2idx_gt = name_to_joint_index(robot_gt)
name2idx_pd = name_to_joint_index(robot_pd)
for jn in LEFT_ARM_JOINT_NAMES + RIGHT_ARM_JOINT_NAMES:
    if jn not in name2idx_gt:
        raise KeyError(f"Joint '{jn}' not found in URDF")

def render_frame(f: int):
    pts_gt, parent_gt = compute_frame_points(
        robot_gt, name2idx_gt, left_gt[f], right_gt[f], LEFT_ARM_JOINT_NAMES, RIGHT_ARM_JOINT_NAMES
    )
    pts_pd, parent_pd = compute_frame_points(
        robot_pd, name2idx_pd, left_pd[f], right_pd[f], LEFT_ARM_JOINT_NAMES, RIGHT_ARM_JOINT_NAMES
    )
    xg, yg, zg = skeleton_lines(pts_gt, parent_gt)
    xp, yp, zp = skeleton_lines(pts_pd, parent_pd)
    return pts_gt, (xg, yg, zg), pts_pd, (xp, yp, zp)

pts_gt0, l_gt0, pts_pd0, l_pd0 = render_frame(0)

fig = go.FigureWidget()
fig.add_trace(go.Scatter3d(x=pts_gt0[:,0], y=pts_gt0[:,1], z=pts_gt0[:,2], mode='markers', marker=dict(size=3, color='royalblue'), name='gt_points'))
fig.add_trace(go.Scatter3d(x=l_gt0[0], y=l_gt0[1], z=l_gt0[2], mode='lines', line=dict(width=5, color='royalblue'), name='gt_bones'))
fig.add_trace(go.Scatter3d(x=pts_pd0[:,0], y=pts_pd0[:,1], z=pts_pd0[:,2], mode='markers', marker=dict(size=3, color='orangered'), name='pred_points'))
fig.add_trace(go.Scatter3d(x=l_pd0[0], y=l_pd0[1], z=l_pd0[2], mode='lines', line=dict(width=5, color='orangered'), name='pred_bones'))

fig.update_layout(
    title=f'ALOHA FK compare  frame=0  |  GT=blue  Pred=orange  offsetY={PRED_OFFSET_Y_M:.3f}m',
    scene=dict(aspectmode='cube'), margin=dict(l=0, r=0, t=40, b=0), height=760, legend=dict(orientation='h')
)

slider = widgets.IntSlider(value=0, min=0, max=T-1, step=1, description='frame', continuous_update=False)
play = widgets.Play(value=0, min=0, max=T-1, step=1, interval=PLAY_INTERVAL_MS, description='play')
widgets.jslink((play, 'value'), (slider, 'value'))


def _on_frame(change):
    f = int(change['new'])
    pts_gt, l_gt, pts_pd, l_pd = render_frame(f)
    with fig.batch_update():
        fig.data[0].x, fig.data[0].y, fig.data[0].z = pts_gt[:,0], pts_gt[:,1], pts_gt[:,2]
        fig.data[1].x, fig.data[1].y, fig.data[1].z = l_gt[0], l_gt[1], l_gt[2]
        fig.data[2].x, fig.data[2].y, fig.data[2].z = pts_pd[:,0], pts_pd[:,1], pts_pd[:,2]
        fig.data[3].x, fig.data[3].y, fig.data[3].z = l_pd[0], l_pd[1], l_pd[2]
        fig.update_layout(title=f'ALOHA FK compare  frame={f}  |  GT=blue  Pred=orange  offsetY={PRED_OFFSET_Y_M:.3f}m')

slider.observe(_on_frame, names='value')
display(widgets.VBox([widgets.HBox([play, slider]), fig]))



GT steps=144  Pred steps=144  display T=144
b3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
footprintb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
base_linkb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
footprintb3Warning[examples/Importers/ImportURDFDemo/BulletUrdfImporter.cpp,126]:
No inertial data for link, using mass=1, localinertiadiagonal = 1,1,1, identity local inertial frameb3Warning[examples